In [13]:
import re 

import pandas as pd 

In [14]:
df = pd.read_csv("data/Train.csv")
print(df.shape)
df.head()

(3081, 8)


,ID,text,Qatar Related,Geography,Politics & Conflict,Health & Wellbeing,Science,Sports
0,fL8IqbB5haFSy2gk,كشف مسؤول رياضي مصري أن الميدالية الذهبية التي...,1,G-7,NO_POLITICS,NO_HEALTH,NO_SCIENCE,SP-3
1,Jivkt5fbyLOBsaQd,أشاد وزير الدولة للشؤون الخارجية لدولة قطر، سل...,1,G-8,PC-1,NO_HEALTH,NO_SCIENCE,NO_SPORTS
2,3m9k8ipWupvzY9VG,بدأ نجوم السينما والتلفزيون وضيوف الحفل السنوي...,1,G-7,PC-1,NO_HEALTH,NO_SCIENCE,SP-2
3,kIuSWafG6YlQSoiA,ونقلت فرانس برس عن إنفانتينو قوله نحن في مرحلة...,0,G-2,NO_POLITICS,NO_HEALTH,NO_SCIENCE,SP-2
4,L1tA6JMVJ7fbMGKm,اعلنت وزارة الداخلية القطرية، الأربعاء، أن الت...,1,G-4,PC-1,NO_HEALTH,SC-3,NO_SPORTS


In [15]:
junk = df["text"].str.contains("urn:schemas", na=False)
print(junk.sum())
df[junk]["text"].str[:80].head()


0


Series([], Name: text, dtype: str)

In [16]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print(model.max_seq_length)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

128


In [17]:
print(model)

vec = model.encode("السلام عليكم")
print(vec.shape)
print(vec[:5])

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
)
(384,)
[ 0.10410672  0.5721247  -0.07995212  0.05652583 -0.14723913]


In [18]:
from sentence_transformers import util

a = model.encode("أنا سعيد جداً")
b = model.encode("أشعر بالفرح")
c = model.encode("السيارة معطلة")

print(util.cos_sim(a, b))
print(util.cos_sim(a, c))

tensor([[0.9252]])
tensor([[-0.0375]])


In [19]:
tokenizer = model.tokenizer
lengths = df["text"].apply(lambda t: len(tokenizer.tokenize(t)))
print(lengths.describe())
print((lengths > model.max_seq_length).mean())

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (444 > 128). Running this sequence through the model will result in indexing errors


count    3081.000000
mean      423.959104
std       326.442733
min        26.000000
25%       242.000000
50%       343.000000
75%       504.000000
max      5015.000000
Name: text, dtype: float64
0.9500162284972412


In [20]:
MAX_TOKENS = 120

def chunk_text(text, max_tokens=MAX_TOKENS):
    units = [u.strip() for u in re.split(r'(?<=[.!?؟])\s+|\n+', text) if u.strip()]
    chunks, current, current_len = [], [], 0
    for u in units:
        n = len(tokenizer.tokenize(u))
        if current and current_len + n > max_tokens:
            chunks.append(" ".join(current))
            current, current_len = [], 0
        current.append(u)
        current_len += n
    if current:
        chunks.append(" ".join(current))
    return chunks

rows = []
for _, r in df.iterrows():
    for i, c in enumerate(chunk_text(r["text"])):
        rows.append({"chunk_id": f'{r["ID"]}_{i}', "parent_id": r["ID"], "chunk_text": c})

chunks_df = pd.DataFrame(rows)
print(len(df), len(chunks_df))
print(chunks_df["chunk_id"].is_unique)

too_long = chunks_df["chunk_text"].apply(lambda t: len(tokenizer.tokenize(t)) > MAX_TOKENS)
print(too_long.sum())

3081 14431
True
183
